# Data Challenge 13 — Interpreting Logistic Regression 

**Purpose**  
Apply what you learned about logistic regression interpretation by analyzing NYC Restaurant Inspection data. 
 
You’ll practice interpreting **continuous**, **binary**, and **categorical** predictors, compute **odds ratios**, and assess model accuracy. 

**Learning Goals**
- Convert coefficients to odds ratios using `np.exp()`.  
- Interpret ORs for continuous, binary, and categorical predictors.  
- Use accuracy to assess logistic regression performance.  
- Communicate results clearly and responsibly.  

**Data:** June 1, 2025 - Nov 4, 2025 Restaurant Health Inspection

[Restaurant Health Inspection](https://data.cityofnewyork.us/Health/DOHMH-New-York-City-Restaurant-Inspection-Results/43nn-pn8j/about_data)


## Instructor Guidance

**Hint: Use the Lecture Deck, Canvas Reading, and Docs to help you with the code**

Use this guide live; students implement below.

**Docs (Quick Links)**
- LogisticRegression — https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html  
- accuracy_score — https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html  
- OneHotEncoder — https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html  
- StandardScaler — https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html  
- np.exp — https://numpy.org/doc/stable/reference/generated/numpy.exp.html  

**Pseudocode Plan**

1️⃣ Load cleaned restaurant inspection data from the previous challenge.  
2️⃣ Define target = `IS_A` (1 = Grade A, 0 = otherwise).  
3️⃣ Predictors →  
    • Continuous = `SCORE`  
    • Binary = `CRITICAL_NUM`  
    • Categorical = `BORO`  
4️⃣ Scale continuous variables; encode categorical ones.  
5️⃣ Fit `LogisticRegression`.  
6️⃣ Exponentiate coefficients (np.exp()) → odds ratios.  
7️⃣ Interpret one continuous, one binary, and one categorical coefficient.  
8️⃣ Evaluate accuracy.  
9️⃣ Reflect on scaling choices and communication of odds.  


## You Do — Student Section
Work in pairs. Comment your choices briefly. Keep code simple—only coerce the columns you use.

## Step 1 — Imports and Plot Defaults

In [10]:
import pandas as pd, numpy as np
import statsmodels.api as sm
from pathlib import Path
import seaborn as sns
import matplotlib.pyplot as plt
import dataprep
from sklearn.model_selection import train_test_split
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler , OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

### Step 2 — Load CSV, Create Columns, Preview

- Point to your New York City Restaurant Inspection Data 
- Create the `is_A` and `critical_num` columns like you did in L11 notebook

In [5]:
df = pd.read_csv('/Users/Marcy_Student/Downloads/DOHMH_New_York_City_Restaurant_Inspection_Results_20251104 copy.csv')


df.head()

data = df[['ACTION','VIOLATION CODE','SCORE','GRADE','INSPECTION TYPE','CRITICAL FLAG','BORO']]
df.dtypes
data.dtypes

ACTION              object
VIOLATION CODE      object
SCORE              float64
GRADE               object
INSPECTION TYPE     object
CRITICAL FLAG       object
BORO                object
dtype: object

## Step 3 — Define Predictors & Target

- Target is `is_A` 
- X predictors are: SCORE, CRITICAL_NUM (created in Step 2), BORO


In [6]:
# Dropping rows with missing SCORE 
data = data.dropna(subset=['SCORE'])

# Filling missing GRADE by mapping SCORE (A:0-13, B:14-27, C:28+) - score found in google
data['SCORE'] = pd.to_numeric(data['SCORE'], errors='coerce')
missing_rows = data['GRADE'].isna()
def score_to_grade(s):
    if pd.isna(s):
        return np.nan
    if s <= 13:
        return 'A'
    if s <= 27:
        return 'B'
    return 'C'

filled = data.loc[missing_rows, 'SCORE'].apply(score_to_grade)
data.loc[missing_rows, 'GRADE'] = filled

data.info()

# feature engineering
data['is_A'] = (data['GRADE'] == 'A').astype(int)
data['critical_num'] = (data['CRITICAL FLAG'] == 'Critical').astype(int)
data.head()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 274939 entries, 18 to 291277
Data columns (total 7 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   ACTION           274939 non-null  object 
 1   VIOLATION CODE   273397 non-null  object 
 2   SCORE            274939 non-null  float64
 3   GRADE            274939 non-null  object 
 4   INSPECTION TYPE  274939 non-null  object 
 5   CRITICAL FLAG    274939 non-null  object 
 6   BORO             274939 non-null  object 
dtypes: float64(1), object(6)
memory usage: 16.8+ MB


,ACTION,VIOLATION CODE,SCORE,GRADE,INSPECTION TYPE,CRITICAL FLAG,BORO,is_A,critical_num
18,Violations were cited in the following area(s).,04L,13.0000,A,Cycle Inspection / Initial Inspection,Critical,Brooklyn,1,1
19,No violations were recorded at the time of thi...,NaN,0.0000,A,Inter-Agency Task Force / Initial Inspection,Not Applicable,Brooklyn,1,0
36,Violations were cited in the following area(s).,08A,13.0000,A,Pre-permit (Operational) / Initial Inspection,Not Critical,Manhattan,1,0
37,Establishment re-opened by DOHMH.,NaN,0.0000,P,Cycle Inspection / Reopening Inspection,Not Applicable,Manhattan,0,0
54,No violations were recorded at the time of thi...,NaN,0.0000,A,Cycle Inspection / Initial Inspection,Not Applicable,Brooklyn,1,0


In [11]:
x1 = data['SCORE']
x2 = data['critical_num']
x3 = data ['BORO']
y = data['is_A']

Xs = data[['SCORE','critical_num','BORO']]

## Step 4 — Split Data (70/30 Stratify by Target)

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    Xs, y,
    test_size=0.3,
    random_state= 42,
    shuffle = True,
    stratify = y
)

## Step 5 – Preprocessing (You can chose to do this in a Pipeline)  

- Scale continuous features  
- Pass binary as is  
- One-hot encode categorical feature (`BORO`)  

In [51]:
# scaling only the continuous variable, SCORE, leaving critical_num as it's a bool
preprocessing = ColumnTransformer(
    transformers=[
        ('numerical', StandardScaler(), ['SCORE']),
        ('categorical', OneHotEncoder(handle_unknown='ignore',drop='first'), ['BORO'])
        ],
    remainder='passthrough'  # leaves critical_num, no scaling needed
)

my_pipeline = Pipeline(steps=[
    ('Process', preprocessing),
    ('logistic_model',LogisticRegression())
])

display(my_pipeline)

Pipeline(steps=[('Process',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numerical', StandardScaler(),
                                                  ['SCORE']),
                                                 ('categorical',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['BORO'])])),
                ('logistic_model', LogisticRegression())])

## Step 6 – Fit Model & Evaluate Accuracy

- Fit `is_A ~ score` using **LogisticRegression**  
- Compute predictions with `.predict()`  
- Evaluate accuracy with `accuracy_score()`

In [53]:
# fitting the model
my_pipeline.fit(X_train, y_train)

#predictions
y_prediction = my_pipeline.predict(X_test)
print(y_prediction)

# extracting the model_steps to access the intercept and the coefficients
model = my_pipeline.named_steps['logistic_model']

print(f'coefficients: {model.coef_}')
print(f'intercept: {model.intercept_}')


np.exp(model.intercept_)

[0 0 1 ... 0 1 1]
coefficients: [[-1.53447747e+01 -9.53038525e-02 -3.99740419e-02 -5.01335594e-03
  -1.19154302e-01  1.02727035e-01]]
intercept: [-8.20072951]


array([0.00027445])

## Step 7 – Extract Coefficients and Convert to Odds Ratios


In [54]:
# Creating a dataframe to see the coeffs and their corresponding features 
# Getting feature names from the preprocessing pipeline, using a method called get_features_names_out()
feature_names = preprocessing.get_feature_names_out()
coefs = model.coef_[0] # make it one array
odds_ratios = np.exp(coefs)

coeffs_df = pd.DataFrame({
    'Variable': feature_names,
    'Coefficient (log-odds)': coefs,
    'Odds Ratio': odds_ratios
})

print(coeffs_df)

print("Accuracy:", accuracy_score(y_test, y_prediction))

                          Variable  Coefficient (log-odds)  Odds Ratio
0                 numerical__SCORE                -15.3448      0.0000
1       categorical__BORO_Brooklyn                 -0.0953      0.9091
2      categorical__BORO_Manhattan                 -0.0400      0.9608
3         categorical__BORO_Queens                 -0.0050      0.9950
4  categorical__BORO_Staten Island                 -0.1192      0.8877
5          remainder__critical_num                  0.1027      1.1082
Accuracy: 0.9826265124512015


## Step 8 – Interpret Each Predictor 

**Remember**
💡 OR > 1 → increases odds of Grade A  
💡 OR < 1 → decreases odds of Grade A

**Type markdown interpreting all 3 predictors in plain english**


In [55]:
data

,ACTION,VIOLATION CODE,SCORE,GRADE,INSPECTION TYPE,CRITICAL FLAG,BORO,is_A,critical_num
18,Violations were cited in the following area(s).,04L,13.0000,A,Cycle Inspection / Initial Inspection,Critical,Brooklyn,1,1
19,No violations were recorded at the time of thi...,NaN,0.0000,A,Inter-Agency Task Force / Initial Inspection,Not Applicable,Brooklyn,1,0
36,Violations were cited in the following area(s).,08A,13.0000,A,Pre-permit (Operational) / Initial Inspection,Not Critical,Manhattan,1,0
37,Establishment re-opened by DOHMH.,NaN,0.0000,P,Cycle Inspection / Reopening Inspection,Not Applicable,Manhattan,0,0
54,No violations were recorded at the time of thi...,NaN,0.0000,A,Cycle Inspection / Initial Inspection,Not Applicable,Brooklyn,1,0
...,...,...,...,...,...,...,...,...,...
291273,No violations were recorded at the time of thi...,NaN,0.0000,A,Cycle Inspection / Initial Inspection,Not Applicable,Brooklyn,1,0
291274,Violations were cited in the following area(s).,04L,40.0000,C,Cycle Inspection / Initial Inspection,Critical,Manhattan,0,1
291275,Violations were cited in the following area(s).,04L,27.0000,B,Cycle Inspection / Re-inspection,Critical,Brooklyn,0,1
291276,Violations were cited in the following area(s).,10B,31.0000,N,Pre-permit (Operational) / Initial Inspection,Not Critical,Brooklyn,0,0


**SCORE**: For every additional score point, the odds of getting an A is multiplied by approximately 0.00, this suggests that a single point does not have a bigger impact on getting an A, the impact is very little, surely because getting an B or C does not come with a point, but multiple points
**BORO_BROOKLYN**: the odds of getting an A in brooklyn is 0.90 times than the baseline, which is 10% decrease compared to the baseline
**BORO_MANHATTAN** : the odds of getting an A in Manhattan is 0.96times than the baseline, which is 4% decrease compared to the baseline
**BORO_QUEENS** : the odds of getting an A in the Queeens is 0.99times than the baseline, which is 1% decrease compared to the baseline
**BORO_STATEN_ISLAND**: the odds of getting an A in Staten Island is 0.88 times than the baseline, which is a 12% decrease
**CRITICAL_NUM**: the odds of getting an A if the restaurant is critical_num is 1.10 times, a 10 % increase compared to the baseline

# We Share — Reflection & Wrap-Up

Write **one short paragraphs** (4–6 sentences). Be specific and use evidence from your notebook.

**Which predictor had the strongest relationship with getting an A grade?**  
Use the odds ratios and accuracy to support your answer.  

- Looking at the odds ration, we notice that the critical num has a bigger odd ratio, suggesting that you are more likely to get an A, if you are critical, which is confusing, because a restaurant can still be critical an have an A, what relationship do we have between critical and grade, are you more likely do get an A if you are critical ? this is something that I would look into.
- The model has a pretty high accuracy, 98%.
- 

